# Rag-AI-agent

## Date

In [1]:
import datetime
import json
import urllib.request

now = datetime.datetime.now()
current_date = now.strftime("%Y-%m-%d")
current_time = now.strftime("%H:%M:%S")

print(f"Date: {current_date}")
print(f"Time: {current_time}")

Date: 2026-10-13
Time: 10:31:00


## Location

In [2]:
headers = {"User-Agent": "Mozilla/5.0"}

# Step 1: Fetch public IP address
ip_url = "https://api.ipify.org?format=json"
ip_req = urllib.request.Request(ip_url, headers=headers)

with urllib.request.urlopen(ip_req) as response:
    client_ip = json.loads(response.read().decode())["ip"]

# Step 2: Geolocation lookup via ip-api.com using the detected IP
geo_url = f"http://ip-api.com/json/{client_ip}"
geo_req = urllib.request.Request(geo_url, headers=headers)

with urllib.request.urlopen(geo_req) as response:
    geo_data = json.loads(response.read().decode())

city = geo_data.get("city", "Unknown City")
region = geo_data.get("regionName", "Unknown Region")
country = geo_data.get("country", "Unknown Country")
lat = geo_data.get("lat")
lon = geo_data.get("lon")
print(f"Detected Location: {city}, {region}, {country}")
print(f"Coordinates: Latitude {lat}, Longitude {lon}")

Detected Location: Lucknow, Uttar Pradesh, India
Coordinates: Latitude 26.8373, Longitude 80.9165


## Current Weather Data

In [3]:
# Step : Query weather data from Open-Meteo if coordinates are available
if lat is not None and lon is not None:
    openmeteo_url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        f"&current_weather=true"
        f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum"
        f"&timezone=auto"
    )

    om_req = urllib.request.Request(openmeteo_url, headers=headers)

    with urllib.request.urlopen(om_req) as om_response:
        weather_data = json.loads(om_response.read().decode())

    current = weather_data.get("current_weather", {})
    daily = weather_data.get("daily", {})

    today_date = daily.get("time", ["Today"])[0]

    print(f"--- Weather Forecast for Today ({today_date}) ---")
    print(f"Current Temperature: {current.get('temperature')}°C")
    print(f"Wind Speed: {current.get('windspeed')} km/h")
    print(
        f"Today's Max Temperature: {daily.get('temperature_2m_max', [None])[0]}°C"
    )
    print(
        f"Today's Min Temperature: {daily.get('temperature_2m_min', [None])[0]}°C"
    )
    print(
        f"Expected Precipitation: {daily.get('precipitation_sum', [None])[0]} mm"
    )
else:
    print("Could not retrieve latitude and longitude to fetch weather data.")

--- Weather Forecast for Today (2026-10-13) ---
Current Temperature: 29.4°C
Wind Speed: 2.2 km/h
Today's Max Temperature: 31.1°C
Today's Min Temperature: 22.9°C
Expected Precipitation: 0.0 mm


## Tavily api for getting historic data of current date wrt india

In [4]:
!pip install langchain_tavily

In [5]:
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 1. Instantiate the search tool
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    include_answer=True,
)

def search_india_history_on_date(day: int, month_name: str, year: int = None, keywords: str = "major historic events facts"):
    """
    Searches historical information in India for a specific day and month.
    Includes target year in query string rather than API publication date filters.
    """
    if year:
        # Include year directly in natural language query
        query = f"What historical events happened in India on {day} {month_name} {year}? {keywords}"
    else:
        # Query across all historical years
        query = f"What happened on {day} {month_name} in Indian history birth anniversaries major events {keywords}"

    # Clean invocation without start_date / end_date restrictions
    results = tavily_search_tool.invoke({
        "query": query
    })

    return results


if __name__ == "__main__":
    '''# Extract current date using your existing logic
    now = datetime.datetime.now()
    extracted_day = now.day
    extracted_month = now.strftime("%B")

    print(f"Extracted Date from System: {extracted_day} {extracted_month}\n")'''

    # Example 1: Search across all historical years for current day/month
    print(f"=== Historical occurrences on {now.day} {now.strftime("%B")} (All Years) ===")
    all_year_results = search_india_history_on_date(
        day=now.day, 
        month_name=now.strftime('%B')
    )
    print(json.dumps(all_year_results, indent=2))

    # Example 2: Search specifically for 2 October 1904 (Lal Bahadur Shastri's birth year)
    print("\n=== Search specifically for 2 October 1904 ===")
    specific_year_results = search_india_history_on_date(
        day=2, 
        month_name="October", 
        year=1904
    )
    print(json.dumps(specific_year_results, indent=2))

=== Historical occurrences on 13 October (All Years) ===
{
  "query": "What happened on 13 October in Indian history birth anniversaries major events major historic events facts",
  "follow_up_questions": null,
  "answer": "On October 13, 1773, French astronomer Charles Messier discovered the Whirlpool Galaxy. In India, October 13 marks the birth anniversary of Bhulabhai Desai and the death anniversary of Kishore Kumar. In world history, October 13 is also noted for the mass arrests of the Knights Templar in 1307 and the laying of the White House cornerstone in 1792. Additionally, notable figures born on this day include Lenny Bruce, Paul Simon, and Sacha Baron Cohen.",
  "images": [],
  "results": [
    {
      "url": "https://www.jagranjosh.com/general-knowledge/this-day-in-history-13-october-1820003270-1",
      "title": "This Day in History: What Happened Today in History \u2013 13 October",
      "content": "# This Day in History: What Happened Today in History \u2013 13 October\n

## previous day news using tavily api

In [6]:
def fetch_previous_day_news(
    day: int,
    month: str | int,
    year: int,
    keywords: str = "India top news headlines politics business sports",
) -> dict:
    """Calculates the previous day relative to the input date (day-month-year)

    and fetches news/headlines for that exact date using TavilySearch.
    """
    # Parse month input into integer
    if isinstance(month, str):
        month_num = datetime.datetime.strptime(month[:3], "%b").month
    else:
        month_num = month

    # Build target date and calculate previous day (T - 1)
    target_date = datetime.date(year, month_num, day)
    previous_day_date = target_date - datetime.timedelta(days=1)

    formatted_prev_date = previous_day_date.strftime("%d %B %Y")

    print(
        f"Input Date: {target_date.strftime('%d %B %Y')} | Fetching Previous Day News for: {formatted_prev_date}\n"
    )

    # Search query explicitly targetting the previous day's date
    query = f"What were the major news, headlines, and events in India on {formatted_prev_date}? {keywords}"

    # Invoke TavilySearch tool
    results = tavily_search_tool.invoke({"query": query})
    return results


if __name__ == "__main__":
    # Example: If current date is 4 October 2026, passing this will fetch news for 3 October 2026
    current_day = 4
    current_month = "October"
    current_year = 2026

    # Fetch news for 3 October 2026
    previous_day_news = fetch_previous_day_news(
        day=current_day,
        month=current_month,
        year=current_year,
    )

    print("=== PREVIOUS DAY NEWS RESULTS (JSON) ===")
    print(json.dumps(previous_day_news, indent=2, ensure_ascii=False))

Input Date: 04 October 2026 | Fetching Previous Day News for: 03 October 2026

=== PREVIOUS DAY NEWS RESULTS (JSON) ===
{
  "query": "What were the major news, headlines, and events in India on 03 October 2026? India top news headlines politics business sports",
  "follow_up_questions": null,
  "answer": "On October 3, 2026, major news in India included a BJP delegation seeking action against Kerala Sports Minister OJ Janeesh for a black-flag protest against Union Home Minister Amit Shah, and Shah announcing the deportation of a narco-trafficking kingpin from Turkey. The Indian Express reported on a bail granted to a gym owner arrested for allegedly threatening a businessman. The Hindu highlighted India's gold win in cricket against Pakistan at the Asian Games and a co-pilot's alleged terrorist attempt on a flydubai flight. The Indian Express archive noted a teacher shortage flagged by an AAP MLA and a campaign dragged down by cross-voting in Rajasthan. The Times of India audio briefin

## Mail sending

In [7]:
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.utils import formataddr, formatdate, make_msgid
import os
import smtplib
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configuration
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587
SENDER_EMAIL = os.getenv("gmail")
SENDER_PASSWORD = os.getenv("google_app_pass")
SENDER_NAME = "Warrior Mindset ⚔️"

RECEIVER_EMAIL = "govind51n9h@gmail.com"  # Change to any target email

if not SENDER_EMAIL or not SENDER_PASSWORD:
    raise ValueError(
        "Missing 'gmail' or 'google_app_pass' environment variables."
    )

# Create Message container
message = MIMEMultipart("alternative")
message["From"] = formataddr((SENDER_NAME, SENDER_EMAIL))
message["To"] = RECEIVER_EMAIL
message["Subject"] = "⚔️ Keep Striking: You Are Unstoppable!"
message["Date"] = formatdate(localtime=True)
message["Message-ID"] = make_msgid()

# Plain-text Fallback
text_content = """\
Keep Striking Your Sword! ⚔️

Life will throw challenges, heavy armor, and dark storms at you. 
But remember: keep striking with your sword and prove to the world that life isn't hard enough to let you down!

Stay Relentless. Keep Fighting. Win the Day. 🛡️🔥
"""

# HTML Content with Dark Theme, Gradients, GIF & Emojis
html_content = """\
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <title>Keep Striking</title>
</head>
<body style="margin: 0; padding: 0; background-color: #0f172a; font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif; color: #f8fafc;">
  <table role="presentation" width="100%" border="0" cellspacing="0" cellpadding="0" style="background-color: #0f172a; padding: 40px 10px;">
    <tr>
      <td align="center">
        <!-- Main Card Container -->
        <table role="presentation" width="100%" style="max-width: 600px; background: linear-gradient(145deg, #1e293b, #0f172a); border: 1px solid #334155; border-radius: 16px; overflow: hidden; box-shadow: 0 20px 25px -5px rgba(0,0,0,0.6);">
          
          <!-- Banner Animated GIF -->
          <tr>
            <td align="center" style="padding: 0;">
              <img src="https://media.giphy.com/media/v1.Y2lkPTc5MGI3NjExcWc1bWkxeW0yaWhzc2Y5dzFtdWV5NXZiODcxaG43dWUydndrbHRuaSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3oKIPnAiaMCws8nOsE/giphy.gif" alt="Warrior Sword Strike GIF" width="100%" style="display: block; max-height: 260px; object-fit: cover; border-bottom: 3px solid #e11d48;" />
            </td>
          </tr>

          <!-- Header & Badge -->
          <tr>
            <td style="padding: 30px 30px 10px 30px; text-align: center;">
              <span style="background: linear-gradient(90deg, #f43f5e, #fb7185); color: #ffffff; padding: 6px 14px; border-radius: 9999px; font-size: 11px; font-weight: 800; letter-spacing: 1.5px; text-transform: uppercase;">
                ⚔️ WARRIOR MOTIVATION
              </span>
              <h1 style="margin-top: 20px; font-size: 26px; font-weight: 800; color: #ffffff; line-height: 1.3;">
                Life is Difficult... <br/><span style="color: #fb7185;">Keep Striking With Your Sword! 🗡️🔥</span>
              </h1>
            </td>
          </tr>

          <!-- Main Message Body -->
          <tr>
            <td style="padding: 10px 30px 20px 30px; line-height: 1.8; font-size: 16px; color: #cbd5e1;">
              <p style="margin-bottom: 16px;">
                Hey Champion 👋,
              </p>
              <p style="margin-bottom: 16px;">
                Whenever the road gets steep and the battles feel endless, remember who you are.
              </p>
              
              <!-- Quote Card Box -->
              <div style="background: rgba(244, 63, 94, 0.12); border-left: 4px solid #f43f5e; padding: 20px; border-radius: 10px; margin: 24px 0; color: #fecdd3; font-size: 17px; font-style: italic; font-weight: 600; line-height: 1.6;">
                "Life is difficult, but keep striking with your sword and prove to the world that it is simply <u>NOT hard enough to let you down</u>!" 🛡️💥
              </div>

              <p style="margin-bottom: 20px;">
                Every single swing breaks through the noise. Stay relentless, keep pushing your limits, and conquer every obstacle in your way! 🚀🔥
              </p>
            </td>
          </tr>

          <!-- Action CTA Button -->
          <tr>
            <td align="center" style="padding: 10px 30px 35px 30px;">
              <a href="https://github.com" target="_blank" style="background: linear-gradient(90deg, #e11d48, #f43f5e); color: #ffffff; text-decoration: none; padding: 14px 34px; border-radius: 10px; font-size: 16px; font-weight: 700; display: inline-block; box-shadow: 0 10px 15px -3px rgba(225, 29, 72, 0.5);">
                Conquer Today ⚔️
              </a>
            </td>
          </tr>

          <!-- Footer -->
          <tr>
            <td style="background-color: #0b1120; padding: 20px; text-align: center; border-top: 1px solid #1e293b; font-size: 12px; color: #64748b;">
              <p style="margin: 0;">Sent with 🔥 by your AI Agent • Keep grinding, never back down!</p>
            </td>
          </tr>

        </table>
      </td>
    </tr>
  </table>
</body>
</html>
"""

# Attach Plain-Text and HTML versions
message.attach(MIMEText(text_content, "plain"))
message.attach(MIMEText(html_content, "html"))

# Send Email
server = None
try:
    server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
    server.starttls()
    server.login(SENDER_EMAIL, SENDER_PASSWORD)

    refused = server.sendmail(
        SENDER_EMAIL, [RECEIVER_EMAIL], message.as_string()
    )
    if refused:
        print("Recipient refused:", refused)
    else:
        print(
            f"🔥 Styled Warrior motivation email sent successfully to {RECEIVER_EMAIL}!"
        )
except Exception as e:
    print(f"Error sending email: {e}")
finally:
    if server:
        server.quit()


🔥 Styled Warrior motivation email sent successfully to govind51n9h@gmail.com!


## Vector DB making

In [8]:
import glob
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

DOCS_DIR  = "rag_docs"
INDEX_DIR = "faiss_index"

def load_and_split(docs_dir):
    chunks = []
    for path in glob.glob(os.path.join(docs_dir, "*.txt")):
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        source = os.path.basename(path)
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        current = ""
        for para in paragraphs:
            if len(current) + len(para) + 2 <= 500:
                current = (current + "\n\n" + para).strip()
            else:
                if current:
                    chunks.append(Document(page_content=current, metadata={"source": source}))
                current = para
        if current:
            chunks.append(Document(page_content=current, metadata={"source": source}))
    return chunks

chunks = load_and_split(DOCS_DIR)
print(f"{len(chunks)} chunks")

C:\Users\Govind Singh\AppData\Local\Temp\ipykernel_27108\3565419234.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


8 chunks


In [9]:
import os
from dotenv import load_dotenv

# Load variables from .env file into os.environ
load_dotenv()

# Now GEMINI_API_KEY is automatically accessible
gemini_api_key = os.getenv("GEMINI_API_KEY")
print(gemini_api_key)


AQ.Ab8RN6KM7M2cCcx5YiFPCSV48xhUv91SLUPyn4yjXSyaf5dydQ


In [10]:
#import faiss-cpu
embeddings  = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(INDEX_DIR)
print(f"Index saved to {INDEX_DIR}/")

Index saved to faiss_index/


In [11]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings

INDEX_DIR = "faiss_index"
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# Load your local DB (requires allow_dangerous_deserialization=True for local files)
db = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)

# Query the database
query = "dedication" 
matching_docs = db.similarity_search(query, k=3) # k is the number of chunks to fetch

# Print the content manually
for i, doc in enumerate(matching_docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source')}")
    print(f"Content:\n{doc.page_content}\n")


--- Chunk 1 ---
Source: struggle.txt
Content:
during 1st year , I started with C for DSA , but later I found that cpp is much better so I switched for about 4 months.
I gave about 30 content on codechef to reach 2 star , probably worst candidate
wrt social life , I am introvert and short tempered so I only have a close friend 'ritik sharma' 
I spent about 1 year learning and practicing basic ml then nlp then dl
it took about 2 months to reach my 1st unpaid internship

--- Chunk 2 ---
Source: bad_habit+fortune.txt
Content:
I don't like to take bath each day , my longest non taken bath period was 1 week
my immunity is low , sad thing I suffered worst fever or diahria during end-sem , due to which my cgpa is still below 8
my hair r getting lost
I am not a an athletic person , but a sleepy fatty brat
i have lost almost all hackathons i have participated , from sih , adove india , flipkart grid , amazon ml ...
my linkedin got hacked by some german hacker , later linkedin rejected to give my

## Foolings ---->

In [12]:
import os
import json
import re
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 1. Instantiate the search tool
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    include_answer=True,
)

def search_india_history_on_date(day: int, month_name: str, year: int = None, keywords: str = "major historic events facts"):
    """
    Searches historical information in India for a specific day and month.
    """
    if year:
        query = f"What historical events happened in India on {day} {month_name} {year}? {keywords}"
    else:
        query = f"What happened on {day} {month_name} in Indian history birth anniversaries major events {keywords}"

    results = tavily_search_tool.invoke({"query": query})
    return results

def format_history_output(tavily_response: dict) -> str:
    """
    Parses Tavily output into Fact 1..5 with heading & details, 
    followed by a overall summary.
    """
    results = tavily_response.get("results", [])
    answer = tavily_response.get("answer", "")
    
    facts = []

    # Process search snippets to extract structured items
    for item in results:
        content = item.get("content", "")
        # Split by newline or sub-headings in Tavily output
        lines = [line.strip() for line in content.split("\n") if line.strip()]
        
        for i, line in enumerate(lines):
            # Identify entries that start with a 4-digit year (e.g., 1932 or Oct 8, 1932)
            if re.match(r'^(\d{4}|[A-Za-z]{3} \d{1,2}, \d{4})', line):
                heading = line
                # Grab the next non-empty line as detail if available
                detail = lines[i+1] if i + 1 < len(lines) else line
                facts.append({"heading": heading, "detail": detail})
            elif len(line) > 20 and not line.startswith("#"):
                # Fallback for descriptive historical sentences
                heading = line[:50] + "..." if len(line) > 50 else line
                facts.append({"heading": heading, "detail": line})

            if len(facts) >= 5:
                break
        if len(facts) >= 5:
            break

    # Format the structured text string
    output_lines = []
    
    for idx, fact in enumerate(facts[:5], start=1):
        output_lines.append(f"fact{idx}:")
        output_lines.append(f"- heading: {fact['heading']}")
        output_lines.append(f"- details: {fact['detail']}\n")

    # Add Summary Section at the end
    output_lines.append("summary:")
    if answer:
        output_lines.append(f"{answer}")
    else:
        output_lines.append("Summary derived from historical events recorded on this date.")

    return "\n".join(output_lines)


# Example usage
if __name__ == "__main__":
    raw_results = search_india_history_on_date(day=8, month_name="October")
    formatted_output = format_history_output(raw_results)
    
    print(formatted_output)

fact1:
- heading: Discover what happened in India on this day
- details: Discover what happened in India on this day

fact2:
- heading: 16 policemen are killed when their vehicle is blow...
- details: 16 policemen are killed when their vehicle is blown up by PWG naxalites on the Basaguda-Jagargunda road in Madhya Pradesh's Bastar division.

fact3:
- heading: Oct 8, 1998
- details: Read more

fact4:
- heading: 1992
- details: RBI announces 1\% cut in the lending and deposit rates.

fact5:
- heading: RBI announces 1\% cut in the lending and deposit r...
- details: RBI announces 1\% cut in the lending and deposit rates.

summary:
On 8 October, Indian history marks the establishment of the Indian Air Force in 1932, celebrated as Indian Air Force Day. The day also observes the birth anniversaries of notable figures such as Raaj Kumar, a famous Hindi film actor, and Kavi Santokh Singh, a renowned Sikh scholar and author. Additionally, it is the death anniversary of Jayaprakash Narayan, a pro

In [13]:
import datetime
import json
import os
import re
from dotenv import load_dotenv
from langchain_tavily import TavilySearch

load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    include_answer=True,
)


def fetch_previous_day_news(
    day: int,
    month: str | int,
    year: int,
    keywords: str = "India top news headlines politics business sports",
) -> dict:
    """Calculates the previous day relative to the input date (day-month-year)
    and fetches news/headlines for that exact date using TavilySearch.
    """
    if isinstance(month, str):
        month_num = datetime.datetime.strptime(month[:3], "%b").month
    else:
        month_num = month

    target_date = datetime.date(year, month_num, day)
    previous_day_date = target_date - datetime.timedelta(days=1)
    formatted_prev_date = previous_day_date.strftime("%d %B %Y")

    print(
        f"Input Date: {target_date.strftime('%d %B %Y')} | Fetching Previous Day News for: {formatted_prev_date}\n"
    )

    query = f"What were the major news, headlines, and events in India on {formatted_prev_date}? {keywords}"
    results = tavily_search_tool.invoke({"query": query})
    return results


def format_news_output(tavily_response: dict) -> str:
    """Parses Tavily response into news1..5 with heading & content,
    followed by an overall summary.
    """
    results = tavily_response.get("results", [])
    answer = tavily_response.get("answer", "")

    news_items = []

    for result in results:
        title = result.get("title", "").strip()
        content = result.get("content", "").strip()

        # Clean title by removing site suffixes like "- The Hindu" or "| The Indian Express"
        cleaned_title = re.sub(
            r"\s*[\|-]\s*(The Hindu|The Indian Express|Times of India|Mathrubhumi).*$",
            "",
            title,
            flags=re.IGNORECASE,
        ).strip()

        # Split content into readable chunks/sentences
        sentences = [
            s.strip()
            for s in re.split(r"\n+|\.\s+", content)
            if len(s.strip()) > 30 and not s.startswith("#")
        ]

        if sentences:
            snippet = ". ".join(sentences[:2]) + "."
            news_items.append({"heading": cleaned_title, "content": snippet})

        if len(news_items) >= 5:
            break

    # Construct formatted output string
    output_lines = []

    for idx, item in enumerate(news_items[:5], start=1):
        output_lines.append(f"news{idx}:")
        output_lines.append(f"heading: {item['heading']}")
        output_lines.append(f"content: {item['content']}\n")

    output_lines.append("summary:")
    if answer:
        output_lines.append(answer)
    else:
        output_lines.append(
            "Summary derived from top national, sports, political, and business news updates of the day."
        )

    return "\n".join(output_lines)


if __name__ == "__main__":
    current_day = 4
    current_month = "October"
    current_year = 2026

    # Fetch raw Tavily results
    raw_results = fetch_previous_day_news(
        day=current_day,
        month=current_month,
        year=current_year,
    )

    # Format into news1..news5 and summary
    formatted_output = format_news_output(raw_results)
    print(formatted_output)

Input Date: 04 October 2026 | Fetching Previous Day News for: 03 October 2026

news1:
heading: India News Live Updates 3 October 2026: Nayara Energy hikes petrol prices by Rs 5, diesel by Rs 3 | India News
content: BJP seeks action against Kerala minister over black-flag protest against Amit Shah. A BJP delegation on Saturday met Kerala Governor Rajendra Vishwanath Arlekar and sought action against state Sports Minister OJ Janeesh over a black-flag protest staged against Union Home Minister Amit Shah in Kollam last week.

news2:
heading: The Hindu top news of the day: October 3, 2026
content: Abhishek Sharma was the familiar nemesis upfront while Tilak Varma became a tormentor towards the end as India lived up to its top billing by defending the Asian Games gold medal with a 19-run win over bitter foes Pakistan in the final on Saturday (October 3, 2026). The co-pilot of flydubai flight FZ1073 attempted to carry out a “terrorist” attack during the flight, the United ​Arab Emirates prose